In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.coffee_pipeline;

In [0]:
SOURCE = "/Volumes/workspace/coffee/landing/online_orders"
TARGET = "/Volumes/workspace/coffee/landing/pipeline_orders"

dbutils.fs.mkdirs(TARGET)
for name in ["batch_001.json", "batch_002.json", "batch_003.json", "batch_004.json"]:
    dbutils.fs.cp(f"{SOURCE}/{name}", f"{TARGET}/{name}")

display(dbutils.fs.ls(TARGET))

In [0]:
%sql
SELECT order_id, customer_id, city, channel, _rescued_data
FROM workspace.coffee_pipeline.bronze_online_orders
ORDER BY order_id;

In [0]:
%sql
SELECT 'job' AS built_by, channel, lines, orders, revenue_eur
FROM workspace.coffee.gold_revenue_by_channel
UNION ALL
SELECT 'pipeline', channel, lines, orders, revenue_eur
FROM workspace.coffee_pipeline.gold_revenue_by_channel
ORDER BY channel, built_by;

In [0]:
%sql
SELECT COUNT(*) AS lines, ROUND(SUM(line_amount_eur), 2) AS revenue_eur
FROM workspace.coffee_pipeline.silver_order_lines;

SELECT order_id, product, line_amount_eur
FROM workspace.coffee_pipeline.silver_rejected_lines;

In [0]:
%sql
SELECT COUNT(*) AS lines, ROUND(SUM(line_amount_eur), 2) AS revenue_eur
FROM workspace.coffee_pipeline.silver_order_lines;

SELECT channel, lines, orders, revenue_eur
FROM workspace.coffee_pipeline.gold_revenue_by_channel
ORDER BY channel;

In [0]:
TARGET = "/Volumes/workspace/coffee/landing/pipeline_orders"

batch_005 = ('{"order_id": 209, "customer_id": 16, "city": "Berlin", '
             '"order_ts": "2026-02-05T10:30:00", "channel": "online", '
             '"items": [{"product": "kettle", "qty": 1, "price_eur": 45.0}]}')

dbutils.fs.put(f"{TARGET}/batch_005.json", batch_005, True)

In [0]:
%sql
SELECT current_user();

DESCRIBE SCHEMA EXTENDED workspace.coffee_pipeline;

SHOW GRANTS ON CATALOG workspace;